In [1]:
import pandas as pd
import numpy as np
import tanalysis.traj_analysis as ta
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import os

In [2]:
dirname = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Tracks\excel_tracks\CXCL10_Conc10.xlsx"
savedir = r"C:\Users\pcanaleta\Documents\Cellpose_segmentation\EXP.HD6.Chips\EXP.HD6.1.1.MatekChips_CXCL10\24h\Tracks\excel_tracks\Results"
filter_values = {'track_duration':(0,200000), 'mean_velocity':(0,100), 'total_distance':(0,100000)}
tracks, names = ta.crop_traj(dirname, filter_tracks=False, filter_values=filter_values)

In [3]:
params = ta.fit_APRW(tracks, names, fr'{savedir}\APRW')
ta.sim_APRW(params, tracks, names, 50, 30, 100, savedir=fr'{savedir}\APRW')

In [ ]:
ids = np.unique(params.index.get_level_values(0))
subsamples = 100
repeats = 10
Nmax = 10

#Extract tlag and dimensions from original tracks
track = tracks.loc[ids[0]]
dim = len(track.columns)-1
tlag = np.unique(np.diff(track['time']))[0]
dt = tlag/subsamples
ss = 10 ** np.ceil(np.log10(repeats))
#Simulation of given number of repeats for each track
xys00 = []
count = 0
for id in ids:
    track_params = params.loc[id]
    beta = 1/track_params['P']
    alpha = (track_params['S']**2)*beta
    FR = np.asarray(np.sqrt(alpha*dt)*dt)
    ap = np.array((max(0, 1-dt/track_params['P'].iloc[0]), max(0, 1-dt/track_params['P'].iloc[1])))
    SE = np.asarray(track_params['SE'])
    fnum = Nmax*subsamples
    if dim==3:
        ap = np.append(ap, ap[-1])
        FR = np.append(FR, FR[-1])
        SE = np.append(SE, SE[-1])

    for rep in range(0, repeats):
        dr = np.zeros((fnum, dim))
        xyz = np.zeros((fnum, dim))

        for i in range(0, fnum-1):
            dr[i+1,:] = FR*np.random.randn(3) + ap*dr[i,:]
            xyz[i+1,:] = xyz[i,:] + dr[i,:]
        exyz = xyz[::subsamples, :]
        oxyz = exyz - np.ones((exyz.shape[0], 1))*exyz[0,:]
        oxyz = oxyz + np.random.randn(len(oxyz[:]),dim)*SE #Position noise

        if dim==3:
            theta = np.random.randn(3)*2*np.pi
            Rx = np.array([[1, 0, 0],
                           [np.cos(theta[0]), -np.sin(theta[0]), 0],
                           [0, np.sin(theta[0]), np.cos(theta[0])]])
      
            Ry = np.array([[np.cos(theta[1]), 0, np.sin(theta[1])],
                           [0, 1, 0],
                           [-np.sin(theta[1]), 0, np.cos(theta[1])]])

            Rz = np.array([[np.cos(theta[2]), -np.sin(theta[2]), 0],
                           [np.sin(theta[2]), np.cos(theta[2]), 0],
                           [0, 0, 1]])
            
            Rm = Rx@Ry@Rz
            rxyz = oxyz@Rm
        
        elif dim==2:
            theta = np.random.randn()*2*np.pi
            Rm = np.array([[np.cos(theta), -np.sin(theta)],
                           [np.sin(theta), np.cos(theta)]])
            rxyz = oxyz@Rm
        
        xyss0 = np.column_stack((np.ones(rxyz.shape[0])*count*ss + rep, np.arange(0, len(rxyz))*tlag, rxyz))
        xys00.append(xyss0)
    count += 1
sim_tracks = pd.DataFrame(np.vstack(xys00))
#sim_tracks.to_excel(savenames, sheet_name='sim_APRW', index=False)
    

In [ ]:
ids = np.unique(tracks.index.get_level_values(0))
for id in ids:
    track = tracks.loc[id]
    dt = np.unique(np.diff(track, axis=0)[:,0][0])[0]
    xyz = track.iloc[:,slice(1,None,1)].dropna()
    frames = xyz.index
    dxyz = np.diff(xyz, axis=0)
    U,S,V = np.linalg.svd(dxyz, full_matrices=False)
    xyzrot = xyz@V.T

    msdp0 = ta.ezmsd(xyzrot.iloc[:,0])
    msdnp0 = ta.ezmsd(xyzrot.iloc[:,1])
    if len(xyz.columns) == 3:
        msdnp0 += ta.ezmsd(xyzrot.iloc[:,2])
    
    t = np.arange(1,26,1)
    poptp, _ = curve_fit(ta.tPRW1D, frames[1:]*dt, msdp0[:], p0=(100,0.1,1), bounds=([0, 0, 0], [1000,100/dt,10]), method='trf', maxfev=10000)
    poptnp, _ = curve_fit(ta.tPRW2D, frames[1:]*dt, msdnp0[:], p0=(100,0.1,1), bounds=([0, 0, 0], [1000,100/dt,10]), method='trf', maxfev=10000)

    print(poptp, poptnp)
    index = pd.MultiIndex.from_product([ids,['p', 'np']])
    print(index)

    plt.figure()
    plt.plot(frames[1:]*dt, msdp0[:])
    plt.plot(frames[1:]*dt, ta.tPRW1D(frames[1:]*dt, *poptp))
    plt.plot(frames[1:]*dt, msdnp0[:])
    plt.plot(frames[1:]*dt, ta.tPRW2D(frames[1:]*dt, *poptnp))

    